# AETHER STT — data prep notebook (run on an A100)

Builds the Mimi extraction cache (semantic codes + byte targets) for
`configs/ctc_base.yaml`, mirrors it to Google Drive, and (optionally)
publishes it to the Hugging Face Hub so it can be reused with
`load_dataset(...)` instead of re-running this notebook.

The two GPU-adjacent stages are split into their own cells so each one's
progress/result is unambiguous under a tight time budget (~1.5h on an A100):

1. **Download** — pulls the full configured LibriSpeech splits via HF
   `datasets` (~30GB, disposable) with no Mimi/GPU involvement, then prints
   a per-split summary so you can confirm what actually landed before
   spending GPU time.
2. **Extract** — runs frozen Mimi over the downloaded audio (already on
   local disk, so this stage is pure GPU throughput, no download stalls)
   to build the (semantic_codes, byte_target) cache (~100-150MB) and
   mirrors it to Drive.
3. **Publish (optional)** — pushes the extracted cache to a Hugging Face
   Hub dataset repo.

Mimi itself is a small model (extraction doesn't need a strong GPU — a T4
is enough for stage 2), but `configs/ctc_base.yaml`'s
`extraction_batch_size=64` is already tuned for an A100, matching how this
notebook is meant to run. Once stage 2 finishes, `train_ctc.ipynb` just
restores the cache from Drive in seconds — no need to re-run this notebook
unless `configs/ctc_base.yaml`'s data settings change (a mismatched
fingerprint makes `prepare_cache` raise rather than silently reusing a
stale cache, so you'll know if a re-run is needed).

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/karl4th/aether-v3.git"
REPO_NAME = "aether-v3"

# Idempotent regardless of how many times this cell is re-run in the same
# kernel session: after the first run cwd is already inside the repo (from
# os.chdir below), so checking os.path.isdir("aether-v3") relative to cwd
# would look one level too deep and clone a second copy nested inside the
# first - repeatable indefinitely. Instead, explicitly handle "already
# standing inside the repo" as its own case.
cwd = Path.cwd()
if cwd.name == REPO_NAME and (cwd / ".git").is_dir():
    subprocess.run(["git", "pull"], check=True, cwd=cwd)
    repo_dir = cwd
    print("Already inside the repo, pulled.")
else:
    repo_dir = cwd / REPO_NAME
    if (repo_dir / ".git").is_dir():
        subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)
        print("Pulled")
    elif repo_dir.exists():
        raise RuntimeError(
            f"{repo_dir} exists but isn't a git checkout (no .git/) - "
            "remove or rename it manually before re-running this cell."
        )
    else:
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
        print("Cloned")
    os.chdir(repo_dir)

print("cwd:", os.getcwd())

In [ ]:
import importlib.util
import subprocess
import sys


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


# Most GPU notebook images already ship a CUDA-matched torch build — don't
# clobber it. Only install if genuinely missing.
if importlib.util.find_spec("torch") is None:
    pip_install("torch")
if importlib.util.find_spec("torchaudio") is None:
    pip_install("torchaudio")

pip_install(
    "transformers>=5.17",  # <5.17 lacks MimiModel.get_audio_codes_mask - see mimi_wrapper.py
    "datasets>=2.19,<4.0",  # >=4.0 requires torchcodec + system ffmpeg for Audio decoding
    "soundfile",
    "librosa",  # datasets<4.0's Audio decode path needs this alongside soundfile
    "pyyaml",
    "numpy",
    "tqdm",
    "huggingface_hub",  # only needed for the optional HF Hub publish step at the end
    "hf_transfer",  # multi-connection Rust downloader - big speedup for the Step 1 download
)

In [ ]:
import logging
import os
import sys

# Must be set before `datasets`/`huggingface_hub` get imported anywhere in
# this kernel (they read this env var once, at import time) - switches HF
# Hub file downloads from a single-connection request to `hf_transfer`'s
# parallel-chunk Rust downloader, which is the main lever for cutting down
# Step 1's ~30GB download time. Requires the `hf_transfer` package (pip
# cell above installs it); silently ignored if it isn't present.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

# So download_raw_splits'/prepare_cache's own progress logs (example
# counts, examples/s, ETA) actually show up in this notebook's output -
# without this, INFO-level logs are silently dropped by the default root
# logger config.
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("using device:", DEVICE)
assert DEVICE == "cuda", "No GPU visible on this VM - check the runtime's accelerator (A100)."


## Google Drive cache

Mounts Drive and mirrors just the small extracted cache there (not the raw
audio) around the `prepare_cache` call below - so `train_ctc.ipynb` never
has to redo this step on a more expensive GPU tier.

In [ ]:
import os
from pathlib import Path

from aether_v3.config import load_config

real_config = load_config("configs/ctc_base.yaml")

try:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/aether-v3")
except ImportError:
    DRIVE_ROOT = Path("drive_cache").resolve()
    print(f"Not running in Colab - falling back to local '{DRIVE_ROOT}' (no cross-session persistence).")

DRIVE_CACHE_DIR = DRIVE_ROOT / "data_cache" / Path(real_config.data.cache_dir).name

# Match extraction parallelism to whatever this VM actually has.
real_config.data.extraction_num_workers = max(2, min(os.cpu_count() or 4, 16))

print("Drive cache dir:", DRIVE_CACHE_DIR)

## Step 1 — Download raw audio (no GPU work yet)

Downloads every configured split (train/validation/test) via HF
`datasets`, with real progress bars showing an actual total (e.g.
"28539/28539") — unlike the Mimi-extraction stage below, whose generator
can't report a total ahead of time. Splits already present in the local HF
`datasets` cache are an instant no-op, so this cell is always safe to
re-run.

`HF_HUB_ENABLE_HF_TRANSFER=1` (set two cells up) switches every HF Hub
file transfer here to `hf_transfer`'s parallel-chunk downloader instead of
one connection per file - this is the main speed lever for the ~30GB
download and needs no other change to run faster.

Per `configs/ctc_base.yaml`: `train.100` (~6GB) + `train.360` (~24GB) for
training, plus `dev-clean`/`test-clean` for eval — roughly **30-35GB of
raw audio** total. Confirm the summary below looks right before moving on
to Mimi extraction.

In [ ]:
from aether_v3.data.librispeech import load_splits
from aether_v3.data.mimi_cache import download_raw_splits

download_raw_splits(real_config)

print("\nDownload summary:")
role_to_specs = {
    "train": real_config.data.train_splits,
    "validation": real_config.data.validation_splits,
    "test": real_config.data.test_splits,
}
for role, specs in role_to_specs.items():
    # Already downloaded/built above, so this is an instant local-cache hit,
    # not a second download - just here to report what's actually on disk.
    raw = load_splits(specs, real_config.data.dataset_id, real_config.data.fallback_dataset_id)
    sample = raw[0]
    sample_duration = len(sample["audio"]["array"]) / sample["audio"]["sampling_rate"]
    print(
        f"  {role:<10} splits={specs} -> {len(raw)} examples "
        f"(sample: {sample_duration:.1f}s, text={sample['text'][:60]!r})"
    )

## Step 2 — Extract Mimi codes (GPU)

Restores any cache role already pushed to Drive (skips straight to
"Done" for those), then extracts whatever's left. Since Step 1 already
downloaded everything, this stage is pure GPU throughput end-to-end - no
download stalls in the middle of the progress log.

If you want to sanity-check on less data/compute before committing to the
full 460h, edit `configs/ctc_base.yaml`'s `data.train_splits` down to just
`["clean/train.100"]` before running this cell (and delete both the local
`data_cache/ctc_base/` and its Drive mirror if you already ran it with the
larger split — a mismatched fingerprint makes `prepare_cache` raise rather
than silently reusing it).

In [ ]:
from aether_v3.data.cache_sync import hydrate_from_remote, push_to_remote
from aether_v3.data.mimi_cache import prepare_cache

restored = hydrate_from_remote(real_config.data.cache_dir, DRIVE_CACHE_DIR)
if restored:
    print(f"Restored from Drive, skipping re-extraction for: {restored}")

prepare_cache(real_config, device=DEVICE)

pushed = push_to_remote(real_config.data.cache_dir, DRIVE_CACHE_DIR)
if pushed:
    print(f"Pushed newly extracted cache to Drive: {pushed}")

## Step 3 (optional) — Publish the extracted cache to the Hugging Face Hub

Wraps the three cached roles (`train`/`validation`/`test`) into one
`DatasetDict` and pushes it to a dataset repo, so it can be pulled with
`load_dataset(HF_REPO_ID)` instead of re-running this notebook.

Set `HF_REPO_ID` below before running. Needs a Hugging Face token with
write access to that repo - set an `HF_TOKEN` env var beforehand, or leave
it unset and `login()` will prompt for one interactively.

In [ ]:
import os
from pathlib import Path

from datasets import DatasetDict, load_from_disk
from huggingface_hub import login

HF_REPO_ID = "your-username/aether-v3-mimi-cache"  # <-- edit before running
HF_PRIVATE = True

token = os.environ.get("HF_TOKEN")
login(token=token) if token else login()

cache_dir = Path(real_config.data.cache_dir)
ds_dict = DatasetDict(
    {role: load_from_disk(str(cache_dir / role)) for role in ("train", "validation", "test")}
)
print(ds_dict)

ds_dict.push_to_hub(HF_REPO_ID, private=HF_PRIVATE)
print(f"Pushed to https://huggingface.co/datasets/{HF_REPO_ID}")

## Done

The cache is now on Drive (and, if you ran Step 3, on the Hugging Face
Hub). You can disconnect this runtime and switch to `train_ctc.ipynb` on a
GPU for the actual training run - it will restore this cache from Drive
(or `load_dataset(HF_REPO_ID)`) in seconds.